# Neural World Model: manual video job

Run an exported Forecast or Backtest job on a GPU machine, then import its result ZIP into the local app. This notebook calls the same package adapter. It receives **only the 17 observed frames**, never the withheld future.

You choose and start the runtime yourself. Check its billing and available GPU first. This notebook does not provision a cloud runtime. The adapter requires CUDA with BF16; a T4 does not satisfy that requirement. No performance claim is made until the hardware test succeeds.

Locally, run this notebook from the source checkout. In Colab, upload the source-only `nwm_colab.zip` created by `python tools/make_colab_bundle.py`. The Files sidebar works if the upload picker is unavailable.

In [ ]:
from pathlib import Path
import os, sys, zipfile
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import files
    source_zip = Path('/content/nwm_colab.zip')
    if not source_zip.exists():
        files.upload()
    if not source_zip.exists():
        raise FileNotFoundError('Upload nwm_colab.zip in the Files sidebar, then rerun this cell.')
    project = Path('/content/nwm_source')
    project.mkdir(exist_ok=True)
    with zipfile.ZipFile(source_zip) as archive:
        entries = archive.infolist()
        if len(entries) > 20000 or sum(x.file_size for x in entries) > 200 * 1024**2:
            raise ValueError('Use the source-only bundle, without datasets/checkpoints.')
        for entry in entries:
            destination = (project / entry.filename).resolve()
            if not destination.is_relative_to(project.resolve()) or (entry.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('Unsafe source archive member')
        archive.extractall(project)
    os.chdir(project)
else:
    project = Path.cwd()
    if not (project / 'pyproject.toml').is_file() and (project.parent / 'pyproject.toml').is_file():
        project = project.parent
    if not (project / 'pyproject.toml').is_file():
        raise FileNotFoundError('Open this notebook from the Neural World Model checkout.')
    os.chdir(project)
print('Source:', project)

## Install the pinned adapter and check the GPU
Existing PyTorch is retained. If this check fails, stop and select a supported machine; no model assets have been downloaded.

In [ ]:
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '.[video]'])
import torch
assert torch.cuda.is_available(), 'A CUDA GPU is required'
assert torch.cuda.is_bf16_supported(), 'This GPU does not support the required BF16 adapter'
x = torch.randn(32, 32, device='cuda', dtype=torch.bfloat16, requires_grad=True)
(x @ x.T).float().square().mean().backward()
print(torch.__version__, torch.cuda.get_device_name(), 'forward/backward passed')
del x
torch.cuda.empty_cache()

## Upload one exported job
In the local app, choose **Export notebook job**, prepare the job, then download its ZIP. Set `JOB_ZIP` below or upload that ZIP in Colab. Only `request.json` and `prefix.npz` are accepted.

In [ ]:
JOB_ZIP = ''  # Local path, or leave blank to use the Colab picker
if not JOB_ZIP and IN_COLAB:
    uploaded = files.upload()
    matches = [name for name in uploaded if name.startswith('nwm-job-') and name.endswith('.zip')]
    if len(matches) != 1:
        raise ValueError('Upload exactly one nwm-job-<id>.zip, or set JOB_ZIP explicitly.')
    JOB_ZIP = matches[0]
job_path = Path(JOB_ZIP).expanduser().resolve()
assert job_path.is_file(), 'Set JOB_ZIP to the exported job'
with zipfile.ZipFile(job_path) as archive:
    assert len(archive.infolist()) == 2 and set(archive.namelist()) == {'request.json', 'prefix.npz'}
    assert archive.getinfo('request.json').file_size < 65536
    import json
    request = json.loads(archive.read('request.json'))
print({key: request[key] for key in ('job_id', 'horizon', 'seed', 'model_revision')})

## Explicit model download
Pinned assets total about 23.65 GiB. The check preserves 8 GiB free and a 27 GiB managed quota. Downloading and GPU execution may consume the runtime resources you selected. Set the flag to `True` only when you intend to download. Prediction itself never downloads weights.

In [ ]:
from nwm.video_model import asset_preflight, download_assets
MANAGED_ROOT = Path('/content/nwm_video') if IN_COLAB else project / 'runs/manual_video'
DOWNLOAD_WEIGHTS = False
check = asset_preflight(MANAGED_ROOT)
print(json.dumps(check, indent=2))
if DOWNLOAD_WEIGHTS:
    model_dir = download_assets(MANAGED_ROOT)
else:
    model_dir = MANAGED_ROOT / 'models/ltx-2b-0.9.8'
    if check['missing_files']:
        raise RuntimeError('Weights are absent. Review the storage check and explicitly enable DOWNLOAD_WEIGHTS.')

## Generate and export
The manual run records its source as `manual_notebook`. This does not certify local hardware feasibility. Keep the runtime alive until the result ZIP is saved. Import the result ZIP into the **same waiting job** in the local app; its job ID, prefix checksum, revision, shape and seed are checked before scoring.

In [ ]:
from nwm.video_model import run_video_job
result_path = job_path.with_name('nwm-result-' + request['job_id'] + '.zip')
response = run_video_job(job_path, result_path, model_dir)
print('Saved:', result_path)
print('Prediction checksum:', response['prediction_sha256'])
if IN_COLAB:
    files.download(str(result_path))